# 01 · Grayscale, fixed threshold and morphology

First stage of the exploration.
The question here was: **can the brightness of a pixel alone separate lesion from healthy tissue?**
Every experiment is kept, including the ones that did not work.

**To run it** you need the leaf scans in `data/images/`, named `vir_<id>.jpg` (ids as in
`data/pliman_validation_50_leaves.csv`, which holds the severity measured with Pliman, used as the reference).
The scans and the tables in `data/` are not included in the repository and are available on reasonable request.

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import exposure

IMAGES_DIR = Path("../../data/images")
REFERENCE = (pd.read_csv("../../data/pliman_validation_50_leaves.csv")
               .set_index("fig_id")["pliman_severity"])

BACKGROUND = 255


def read_bgr(leaf_id):
    img = cv2.imread(str(IMAGES_DIR / f"vir_{leaf_id}.jpg"))
    if img is None:
        raise FileNotFoundError(f"vir_{leaf_id}.jpg not found in {IMAGES_DIR}")
    return img


def read_gray(leaf_id):
    return cv2.cvtColor(read_bgr(leaf_id), cv2.COLOR_BGR2GRAY)


def leaf_area(gray):
    """Leaf pixels = everything that is not the pure-white scanner background."""
    return int(np.sum(gray != BACKGROUND))

## 1. One leaf in grey levels

Leaf `vir_46` converted to grey (0 = black, 255 = white) and the histogram of all its pixels. The background is a tall
peak at 255; the leaf is a bell between about 60 and 150.

In [ ]:
gray46 = read_gray(46)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(gray46, cmap="gray"); ax[0].axis("off")
ax[1].hist(gray46.ravel(), bins=32, range=(0, 256))
ax[1].set(xlabel="Brightness", ylabel="Number of pixels", title="Histogram of leaf 46")
plt.show()

## 2. Lesion against healthy tissue

A 60 × 60 patch on a lesion and another on healthy tissue (coordinates chosen by eye on leaf 46), with their
histograms. Healthy tissue is one bell; the lesion shows two bumps, because the pixels at the margin of the lesion
overlap with healthy brightness. No single threshold can separate them perfectly.

In [ ]:
lesion = gray46[60:120, 160:220]
healthy = gray46[310:370, 170:230]

fig, ax = plt.subplots(2, 3, figsize=(13, 7))
for row, (patch, name) in enumerate([(lesion, "Lesion"), (healthy, "Healthy tissue")]):
    ax[row, 0].imshow(patch, cmap="gray", vmin=0, vmax=255); ax[row, 0].set_title(name); ax[row, 0].axis("off")
    ax[row, 1].hist(patch.ravel(), bins=32, range=(0, 256))
    ax[row, 1].set(xlabel="Brightness", ylabel="Pixels", title=f"Histogram: {name.lower()}")
for patch, color, label in [(healthy, "green", "Healthy"), (lesion, "red", "Lesion")]:
    ax[0, 2].hist(patch.ravel(), bins=32, range=(0, 256), color=color, alpha=0.6, label=label)
ax[0, 2].legend(); ax[0, 2].set_title("Both together")
ax[1, 2].axis("off")
plt.tight_layout(); plt.show()

## 3. First model: a fixed threshold

Every pixel at or below 80 is a lesion; severity is lesion pixels over leaf pixels.
Tried first on three leaves, then on all 50 and compared with Pliman.

In [ ]:
def severity_fixed_threshold(gray, threshold=80):
    return 100 * np.sum(gray <= threshold) / leaf_area(gray)


def evaluate(severity_fn, ids=None):
    """Run a severity function on every leaf and put it next to the Pliman reference."""
    ids = list(REFERENCE.index if ids is None else ids)
    df = pd.DataFrame({"pred": {i: severity_fn(read_gray(i)) for i in ids}, "pliman": REFERENCE.loc[ids]})
    df["error"] = df["pred"] - df["pliman"]
    return df


def summary(df):
    return pd.Series({
        "n": len(df),
        "correlation": df["pred"].corr(df["pliman"]),
        "mean_abs_error": df["error"].abs().mean(),
        "mean_difference": df["error"].mean(),
    })


def plot_vs_reference(df, ax, title, label_points=True):
    """Predicted against Pliman. Green: perfect agreement; red: shifted by 2 points on each axis."""
    ax.scatter(df["pred"], df["pliman"])
    if label_points:
        for leaf_id, row in df.iterrows():
            ax.annotate(str(leaf_id), (row["pred"], row["pliman"]), fontsize=7)
    lo, hi = min(df["pred"].min(), df["pliman"].min()), max(df["pred"].max(), df["pliman"].max())
    ax.plot([lo, hi], [lo, hi], "g--")
    ax.plot([lo + 2, hi + 2], [lo, hi], "r--")
    ax.plot([lo, hi], [lo + 2, hi + 2], "r--")
    ax.set(xlabel="Predicted severity (%)", ylabel="Pliman severity (%)", title=title)


for leaf_id in (46, 126, 156):
    print(leaf_id, "predicted:", round(severity_fixed_threshold(read_gray(leaf_id)), 1),
          "| Pliman:", REFERENCE.loc[leaf_id])

In [ ]:
fixed = evaluate(severity_fixed_threshold)
print(summary(fixed).round(2))
fig, ax = plt.subplots(figsize=(6, 5))
plot_vs_reference(fixed, ax, "Fixed threshold (80)")
plt.show()

Result: a lot of dispersion and many false positives (for example, the shadow of the midrib counts as lesion).
The leaves with the largest errors were the ones I looked at next; leaves 134 (a lot of lesion) and 155 (very little)
were taken as typical failures.

## 4. Other ways to threshold (did not work)

* **Otsu's automatic threshold** separates *leaf from background*, not lesion from leaf.
* **Adaptive threshold** (local mean or Gaussian; I tried block size 21/offset 30 and 71/60): it marks the borders of
  the leaf much more than the lesions, whatever block size and offset I used.
* **Edge kernels** (vertical, horizontal and diagonal Prewitt-like kernels) to find borders; shown on leaf 134
  and not followed up.

In [ ]:
def severity_otsu(gray):
    """Otsu threshold over the whole image. Returns severity, the threshold and the binary image."""
    threshold, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_OTSU)
    return 100 * np.sum(binary == 0) / leaf_area(gray), threshold, binary


def severity_adaptive(gray, block_size=21, offset=30):
    """Adaptive threshold, local mean and local Gaussian. Returns both severities and both binary images."""
    by_mean = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, block_size, offset)
    by_gauss = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, block_size, offset)
    area = leaf_area(gray)
    return 100 * np.sum(by_gauss == 0) / area, 100 * np.sum(by_mean == 0) / area, by_gauss, by_mean


sev_otsu, otsu_thr, otsu_bin = severity_otsu(gray46)
sev_gauss, sev_mean, ad_gauss, ad_mean = severity_adaptive(gray46)
print(f"Otsu threshold: {otsu_thr:.0f} -> {sev_otsu:.1f}%   |   adaptive: gauss {sev_gauss:.1f}%, mean {sev_mean:.1f}%")

fig, ax = plt.subplots(1, 3, figsize=(14, 5))
for a, im, t in zip(ax, [otsu_bin, ad_gauss, ad_mean], ["Otsu", "Adaptive (Gaussian)", "Adaptive (mean)"]):
    a.imshow(im, cmap="gray"); a.set_title(t); a.axis("off")
plt.show()

In [ ]:
KERNEL_VERTICAL = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]])
KERNEL_HORIZONTAL = KERNEL_VERTICAL.T
KERNEL_DIAGONAL = np.array([[1, 1, 0], [1, 0, -1], [0, -1, -1]])

g134 = read_gray(134)
edges = {name: cv2.filter2D(g134, -1, k) for name, k in
         [("vertical", KERNEL_VERTICAL), ("horizontal", KERNEL_HORIZONTAL), ("diagonal", KERNEL_DIAGONAL)]}
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(g134, cmap="gray"); ax[0].set_title("Leaf 134")
for a, (name, im) in zip(ax[1:], edges.items()):
    a.imshow(im, cmap="gray"); a.set_title(f"{name} edges")
for a in ax: a.axis("off")
plt.show()

## 5. Clean the threshold with smoothing and morphology

Threshold, then **dilate** the white part (which erases dark specks smaller than the kernel) and **erode** it (which
makes the remaining dark regions grow again). In the binary image lesions are 0, so what OpenCV calls "dilate" shrinks
the lesions and "erode" grows them.

Variants tried:

| Name | Structuring element | Dilate / erode iterations | Threshold |
|---|---|---|---|
| `square_1_3` | square 3 × 3 | 1 / 3 | 80 |
| `ellipse_1_3` | ellipse 3 × 3 | 1 / 3 | 80 (best overall) |
| `ellipse_2_2` | ellipse 3 × 3 | 2 / 2 | 80 (best for low severity, worse for the rest) |
| `ellipse5_2_2` | ellipse 5 × 5 | 2 / 2 | 80 |
| `ellipse_2_2_t65` | ellipse 3 × 3 | 2 / 2 | 65 |
| `square_2_ellipse5_2` | square 3 × 3, then ellipse 5 × 5 | 2 / 2 | 80 |
| `square_1_ellipse5_3` | square 3 × 3, then ellipse 5 × 5 | 1 / 3 | 80 |

I kept the **elliptical structuring element** rather than the square one.

In [ ]:
def structuring_element(shape, size):
    kind = cv2.MORPH_ELLIPSE if shape == "ellipse" else cv2.MORPH_RECT
    return cv2.getStructuringElement(kind, (size, size))


def lesion_mask_morphology(gray, threshold=80, dilate_iters=1, erode_iters=3,
                           dilate_element=("ellipse", 3), erode_element=("ellipse", 3), median_blur=None):
    """Binary image where lesions are 0 (white = background and healthy tissue)."""
    if median_blur:
        gray = cv2.medianBlur(gray, median_blur)
    binary = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)[1]
    binary = cv2.dilate(binary, structuring_element(*dilate_element), iterations=dilate_iters)
    return cv2.erode(binary, structuring_element(*erode_element), iterations=erode_iters)


def severity_morphology(gray, median_blur=None, **kwargs):
    mask = lesion_mask_morphology(gray, median_blur=median_blur, **kwargs)
    area = leaf_area(cv2.medianBlur(gray, median_blur) if median_blur else gray)
    return 100 * np.sum(mask == 0) / area


MORPHOLOGY_VARIANTS = {
    "square_1_3":          dict(dilate_iters=1, erode_iters=3, dilate_element=("square", 3), erode_element=("square", 3)),
    "ellipse_1_3":         dict(dilate_iters=1, erode_iters=3),
    "ellipse_2_2":         dict(dilate_iters=2, erode_iters=2),
    "ellipse5_2_2":        dict(dilate_iters=2, erode_iters=2, dilate_element=("ellipse", 5), erode_element=("ellipse", 5)),
    "ellipse_2_2_t65":     dict(dilate_iters=2, erode_iters=2, threshold=65),
    "square_2_ellipse5_2": dict(dilate_iters=2, erode_iters=2, dilate_element=("square", 3), erode_element=("ellipse", 5)),
    "square_1_ellipse5_3": dict(dilate_iters=1, erode_iters=3, dilate_element=("square", 3), erode_element=("ellipse", 5)),
}

morphology_results = {name: evaluate(lambda g, kw=kw: severity_morphology(g, **kw))
                      for name, kw in MORPHOLOGY_VARIANTS.items()}
print(pd.DataFrame({name: summary(df) for name, df in morphology_results.items()}).T.round(2))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
plot_vs_reference(fixed, ax[0], "Fixed threshold (80)")
plot_vs_reference(morphology_results["ellipse_1_3"], ax[1], "Threshold + morphology (ellipse, 1 / 3)")
plt.show()

**Conclusion of this stage** (also the conclusion of the article): a fixed threshold is not universal, and each image
would need manual adjustment, which defeats the purpose of automating.

## 6. Let the user pick the threshold (semi-automatic)

A histogram helper, and the same morphology function with a threshold chosen by looking at it.
This worked better on difficult leaves, but it is **not automatic**.

* Leaf 155, with little lesion: a threshold of 85 took healthy tissue, and 60 was the best.
* Leaf 14: I first tried brightening it (gamma 0.6) and using a higher threshold (120). Choosing the threshold by
  hand on the original image (60) gave the better value.

In [ ]:
def show_histogram(gray, title="Histogram"):
    fig, ax = plt.subplots(1, 2, figsize=(16, 4))
    ax[0].imshow(gray, cmap="gray"); ax[0].axis("off")
    ax[1].hist(gray.ravel(), bins=64, range=(0, 250))
    ax[1].set_xticks(range(0, 251, 10))
    ax[1].set(xlabel="Brightness", ylabel="Pixels", title=title)
    plt.show()


def show_manual_threshold(leaf_id, threshold):
    """Severity and lesion mask for a threshold chosen by the user."""
    gray = read_gray(leaf_id)
    mask = lesion_mask_morphology(gray, threshold=threshold)
    print(f"leaf {leaf_id}, threshold {threshold}: {100 * np.sum(mask == 0) / leaf_area(gray):.1f}%")
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(cv2.cvtColor(read_bgr(leaf_id), cv2.COLOR_BGR2RGB)); ax[0].set_title("Leaf")
    ax[1].imshow(mask, cmap="gray"); ax[1].set_title("Detected lesions (black)")
    for a in ax: a.axis("off")
    plt.show()


show_histogram(read_gray(155), "Leaf 155")
for threshold in (85, 70, 60):
    show_manual_threshold(155, threshold)

In [ ]:
leaf14 = read_bgr(14)
brightened = exposure.adjust_gamma(leaf14, 0.6)
by_hand = severity_morphology(cv2.cvtColor(leaf14, cv2.COLOR_BGR2GRAY), threshold=60)
by_gamma = severity_morphology(cv2.cvtColor(brightened, cv2.COLOR_BGR2GRAY), threshold=120)
print(f"leaf 14 | threshold by hand (60): {by_hand:.1f}%  |  gamma 0.6 + threshold 120: {by_gamma:.1f}%  |  Pliman: {REFERENCE.loc[14]}%")

## 7. Ideas that were tried and dropped

* **Remove a strip in the middle of the image** to erase the midrib (10 px wide, centred). It assumes the midrib is
  always in the middle of the image, and it was not used afterwards.
* **Erase the veins by hand** on one severe leaf (`vir_100`), only to measure how much of the detected "lesion" was
  vein (two rectangles blanked by hand). Useful to understand the error, not usable as a method.
* **Median blur (kernel 11) before the threshold**, first on leaves 46 and 112 and then on all 50 leaves. The next
  stage kept the blur and moved to colour.

In [ ]:
def blank_central_strip(mask, half_width=5):
    """Whiten a vertical strip around the centre of a lesion mask (tried to remove the midrib)."""
    out = mask.copy()
    centre = out.shape[1] / 2
    out[:, int(centre - half_width):int(centre + half_width)] = 255
    return out


gray155 = read_gray(155)
mask = lesion_mask_morphology(gray155, threshold=60)
with_strip = blank_central_strip(mask)
print("without strip:", round(100 * np.sum(mask == 0) / leaf_area(gray155), 2),
      "| with strip:", round(100 * np.sum(with_strip == 0) / leaf_area(gray155), 2))

In [ ]:
blurred = evaluate(lambda g: severity_morphology(g, median_blur=11, threshold=80))
print(summary(blurred).round(2))

Next: [`02_color_features_pca_kmeans.ipynb`](02_color_features_pca_kmeans.ipynb) — use the colour, not only the brightness.